In [ ]:
import pyproj
pyproj.datadir.set_data_dir("/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/proj/9.2.0/share/proj")

In [ ]:
from eval import *
from utils_cartopy import global_mean
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from eval_shakirova import ALBEDO_KERNEL_PATH, compute_responses
from cartopy.util import add_cyclic_point
import pandas as pd
import matplotlib.ticker as mticker

In [ ]:
torch.set_num_threads(1)

In [ ]:
checkpoint_path = Path('checkpoints/fal/2011-2014_3,6,9,12_sob_fal_clearsky/best_model.pt')
config_path = Path('configs/model/fal/2011-2014_3,6,9,12_sob_fal_clearsky.yaml')
config = load_config(config_path)
vc = variable_config_from_omegaconf(config)
model, preprocessor, _ = load_model_and_preprocessor(config, checkpoint_path)

In [ ]:
def zero_like_response(field):
    return xr.zeros_like(field)


def plot_text_on_ax(ax, text):
    ax.text(
        1.02,
        1,
        text,
        transform=ax.transAxes,
        horizontalalignment="right",
        verticalalignment="bottom",
    )


def add_fields(fields):
    total = fields[0]
    for field in fields[1:]:
        total = total + field
    return total


def shakirova_terms(responses, sky):
    suffix = "all" if sky == "all" else "clr"
    target = responses[f"dR_era5_{suffix}"]
    if sky == "all":
        kernel_parts = [
            responses["dR_a_k_all"],
            responses["dR_q_k_all"],
            responses["dR_c_k_all"],
        ]
        nn_parts = [
            responses["dR_a_nn_all"],
            responses["dR_q_nn_all"],
            responses["dR_c_nn_all"],
        ]
        nn_allcross = responses["dR_nn_all"]
    else:
        kernel_parts = [responses["dR_a_k_clr"], responses["dR_q_k_clr"]]
        nn_parts = [responses["dR_a_nn_clr"], responses["dR_q_nn_clr"]]
        nn_allcross = responses["dR_nn_clr"]

    kernel_sum = add_fields(kernel_parts)
    nn_sum = add_fields(nn_parts)
    return {
        "target": target,
        "kernel": [*kernel_parts, kernel_sum, target - kernel_sum],
        "nn": [*nn_parts, nn_sum, target - nn_sum],
        "allcross": [nn_allcross, target - nn_allcross],
    }


def cyclic_data(data, lon):
    values = data.to_numpy() if hasattr(data, "to_numpy") else np.asarray(data)
    return add_cyclic_point(values, coord=np.asarray(lon), axis=-1)


def add_gridlines_on_ax(ax, i, j, n_rows):
    gl = ax.gridlines(draw_labels=True)

    # Left column only (j == 0)
    gl.left_labels = j == 0
    gl.right_labels = False

    # Bottom row only (i == len(axes) - 1)
    gl.bottom_labels = i == n_rows - 1
    gl.top_labels = False


def plot_shakirova_field(
    ax,
    field,
    title="",
    vmin=-60,
    vmax=60,
    step=5,
    cmap="RdBu_r",
    left_labels=False,
    bottom_labels=False,
    arctic=False,
    titlepad=None,
    extend='both'
):
    plot_data, plot_lon = cyclic_data(field, field.longitude)
    im = ax.contourf(
        plot_lon,
        field.latitude,
        plot_data,
        levels=np.arange(vmin, vmax + step, step),
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extend=extend,
    )
    ax.coastlines()
    if arctic:
        ax.set_extent([-180, 180, 65, 90], ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = left_labels
    gl.bottom_labels = bottom_labels
    gl.xlocator = mticker.FixedLocator([-180, -90, 0, 90, 180])
    #gl.ylocator = mticker.FixedLocator([-60, -30, 0, 30, 60])

    if title:
        ax.set_title(title, loc="left", pad=titlepad)
    return im


def arctic_field(field):
    return field.where(field.latitude >= 65, drop=True)


def response_mean(field, arctic=False):
    return float(global_mean(arctic_field(field) if arctic else field))


def annotate_response(ax, field, residual=False, arctic=False, units = "W m$^{-2}$"):
    field = arctic_field(field) if arctic else field
    if residual:
        mbe = float(global_mean(field))
        rmse = float(np.sqrt(global_mean(field**2)))
        plot_text_on_ax(
            ax, f"MBE: {mbe:.2f} " + units + f" , RMSE: {rmse:.2f} " + units
        )
    else:
        plot_text_on_ax(ax, f"{response_mean(field):.2f} " + units)


def panel_title(label, title):
    return rf"$\bf{{({label})}}$ {title}"


def add_bottom_colorbar(fig, im, axes_list, label, gap=0.01, height=0.035):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bboxes = [
        axis.get_tightbbox(renderer).transformed(fig.transFigure.inverted())
        for axis in axes_list
    ]
    x0 = min(axis.get_position().x0 for axis in axes_list)
    x1 = max(axis.get_position().x1 for axis in axes_list)
    y0 = min(bbox.y0 for bbox in bboxes) - gap - height
    cax = fig.add_axes([x0, y0, x1 - x0, height])
    cax.set_in_layout(False)
    return fig.colorbar(im, cax=cax, orientation="horizontal", label=label)

def add_right_colorbar(fig, im, axes_list, label, gap=0.01, width=0.01):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bboxes = [
        axis.get_tightbbox(renderer).transformed(fig.transFigure.inverted())
        for axis in axes_list
    ]
    y0 = min(axis.get_position().y0 for axis in axes_list)
    y1 = max(axis.get_position().y1 for axis in axes_list)
    x1 = max(bbox.x1 for bbox in bboxes) + gap
    cax = fig.add_axes([x1, y0, width, y1 - y0])
    cax.set_in_layout(False)
    return fig.colorbar(im, cax=cax, orientation="vertical", label=label)

def sum_response_expr(source, sky):
    suffix = "K" if source == "kernel" else "NN"
    subscript = "a+c+q" if sky == "all" else "a+q"
    return rf"\Delta R_{{{subscript}}}^{{{suffix}}}"


def sum_response_title(source, sky):
    return "$" + sum_response_expr(source, sky) + "$"


def make_shakirova_total_figure(sky, arctic=False):
    terms = shakirova_terms(shakirova_responses, sky)
    rows = [
        (terms["target"], None),
        (terms["kernel"][-2], terms["kernel"][-1]),
        (terms["nn"][-2], terms["nn"][-1]),
        (terms["allcross"][0], terms["allcross"][1]),
    ]

    figsize = (11.69, 7) if arctic else (11.69, 3.75)
    fig, ax = plt.subplots(
        2,
        4,
        figsize=figsize,
        subplot_kw=dict(
            projection=(
                ccrs.NorthPolarStereo(central_longitude=0)
                if arctic
                else ccrs.PlateCarree(central_longitude=180)
            )
        ),
        layout="constrained",
    )
    fig.get_layout_engine().set(rect=[0.06, 0.08, 1, 0.94], h_pad=0.01, w_pad=0.01, hspace=0.01, wspace=0.01)

    response_im = residual_im = None
    response_titles = [
        r"$\Delta R$",
        sum_response_title("kernel", sky),
        sum_response_title("nn", sky),
        r"$\Delta R_{allcross}^{NN}$",
    ]
    residual_titles = [
        "",
        r"(a) - (b)",
        r"(a) - (c)",
        r"(a) - (d)",
    ]
    response_labels = list("abcd")
    residual_labels = ["", "e", "f", "g"]
    for col, (response, residual) in enumerate(rows):
        response_im = plot_shakirova_field(
            ax[0][col],
            response,
            panel_title(response_labels[col], response_titles[col]),
            vmin=-60,
            vmax=60,
            step=5,
            left_labels=(col == 0),
            bottom_labels=(col == 0),
            arctic=arctic,
            #titlepad=24,
        )
        annotate_response(ax[0][col], response, arctic=arctic)
        if residual is None:
            ax[1][col].set_visible(False)
            continue
        residual_im = plot_shakirova_field(
            ax[1][col],
            residual,
            panel_title(residual_labels[col], residual_titles[col]),
            vmin=-8,
            vmax=8,
            step=1,
            left_labels=(col == 1),
            bottom_labels=True,
            arctic=arctic,
            #titlepad=24,
        )
        annotate_response(ax[1][col], residual, arctic=arctic)

    fig.canvas.draw()
    #def add_right_colorbar(fig, im, axes_list, label, gap=0.01, width=0.035):
    add_right_colorbar(
        fig, response_im, [ax[0][col] for col in range(4)], "W m$^{-2}$",
    )
    add_right_colorbar(
        fig, residual_im, [ax[1][col] for col in range(1, 4)], "W m$^{-2}$"
    )
    fig.set_layout_engine("none")
    region = "_arctic" if arctic else ""
    fig.savefig(
        shakirova_output_dir / f"shakirova_{sky}_total{region}.png", bbox_inches="tight"
    )
    return fig


def make_shakirova_component_figure(sky, arctic=False):
    terms = shakirova_terms(shakirova_responses, sky)
    if sky == "all":
        inputs = [
            (shakirova_anomaly.fal, r"$\Delta a$", "a", -0.5, 0.5, 0.05, ""),
            (shakirova_anomaly.tcwv, r"$\Delta q$", "q", -20, 20, 2, "kg m$^{-2}$"),
            (shakirova_anomaly_ecod, r"$\Delta c$", "c", -1, 1, 0.1, ""),
            (terms["target"], r"$\Delta R$", "sum", -60, 60, 5, "W m$^{-2}$"),
        ]
    else:
        inputs = [
            (shakirova_anomaly.fal, r"$\Delta a$", "a", -0.5, 0.5, 0.05, ""),
            (shakirova_anomaly.tcwv, r"$\Delta q$", "q", -20, 20, 2, "kg m$^{-2}$"),
            (terms["target"], r"$\Delta R$", "sum", -60, 60, 5, "W m$^{-2}$"),
        ]

    if arctic:
        figsize = (11.69, 10.75) if sky == "all" else (8.27, 9.5)
    else:
        figsize = (11.69, 6) if sky == "all" else (8.27, 6)

    fig, ax = plt.subplots(
        3,
        len(inputs),
        figsize=figsize,
        subplot_kw=dict(
            projection=(
                ccrs.NorthPolarStereo(central_longitude=0)
                if arctic
                else ccrs.PlateCarree(central_longitude=180)
            )
        ),
        layout="constrained",
    )
    fig.get_layout_engine().set(rect=[0.06, 0.09, 1, 0.94], h_pad=0.01, w_pad=0.01, hspace=0.01, wspace=0.01)

    response_im = None
    panel_labels = [list("abcdefghijkl"[row * len(inputs):(row + 1) * len(inputs)]) for row in range(3)]
    for col, (
        input_field,
        input_title,
        response_part,
        vmin,
        vmax,
        step,
        cbar_label,
    ) in enumerate(inputs):
        im = plot_shakirova_field(
            ax[0][col],
            input_field,
            panel_title(panel_labels[0][col], input_title),
            vmin=vmin,
            vmax=vmax,
            step=step,
            left_labels=(col == 0),
            bottom_labels=False,
            arctic=arctic,
        )
        plot_text_on_ax(
            ax[0][col],
            f"{response_mean(input_field, arctic=arctic):.2f}"
            + (f" {cbar_label}" if cbar_label else ""),
        )
        fig.colorbar(im, ax=ax[0][col], orientation="horizontal", label=cbar_label)

        for row, source in enumerate(["kernel", "nn"], start=1):
            field = terms[source][col]
            title = sum_response_title(source, sky) if response_part == "sum" else rf'$\Delta R_{{{response_part}}}^{{{"K" if source == "kernel" else "NN"}}}$'
            response_im = plot_shakirova_field(
                ax[row][col],
                field,
                panel_title(panel_labels[row][col], title),
                vmin=-60,
                vmax=60,
                step=5,
                left_labels=(col == 0),
                bottom_labels=(row == 2),
                arctic=arctic,
            )
            annotate_response(ax[row][col], field, arctic=arctic)

    fig.canvas.draw()
    add_bottom_colorbar(
        fig,
        response_im,
        [ax[row][col] for row in (1, 2) for col in range(len(inputs))],
        "W m$^{-2}$",
    )
    fig.set_layout_engine("none")
    region = "_arctic" if arctic else ""
    fig.savefig(
        shakirova_output_dir / f"shakirova_{sky}_components{region}.png",
        bbox_inches="tight",
    )
    return fig

In [ ]:
years = [1992, 2020]
raw_era5_paths = generate_paths_yearly(config.dataset.era5.raw_path, years, make_era5_filename)
raw = xr.open_mfdataset(raw_era5_paths, combine="nested", concat_dim="date")

In [ ]:
model, preprocessor, _ = load_model_and_preprocessor(config, checkpoint_path, downscaling=False)

shakirova_output_dir = Path('FiguresNeurIPS2')
shakirova_output_dir.mkdir(exist_ok=True, parents=True)
shakirova_base_date = '1992-09'
shakirova_perturbed_date = '2020-09'
response_save_path = shakirova_output_dir / f'saved_responses_no_downscale.nc'

K_a = xr.open_dataset(ALBEDO_KERNEL_PATH).isel(month=8)
shakirova_grid = {'latitude': K_a.latitude, 'longitude': K_a.longitude}
shakirova_ds = raw.interp(**shakirova_grid)
shakirova_base = select_date_state(shakirova_ds, shakirova_base_date)
shakirova_perturbed = select_date_state(shakirova_ds, shakirova_perturbed_date)
shakirova_anomaly = shakirova_perturbed - shakirova_base
shakirova_anomaly_ecod = (
    preprocessor.transform(shakirova_perturbed) - preprocessor.transform(shakirova_base)
).ecod.compute()

if response_save_path.exists():
    shakirova_responses = xr.load_dataset(response_save_path)
else:
    shakirova_responses = compute_responses(
        shakirova_perturbed,
        shakirova_base,
        model,
        preprocessor,
        vc,
        shakirova_base_date,
        shakirova_perturbed_date,
        Path(config.dataset.era5.raw_path),
    )
    shakirova_responses.to_netcdf(response_save_path)


In [ ]:
#fig_shakirova_all_total = make_shakirova_total_figure('all')
#fig_shakirova_clr_total = make_shakirova_total_figure('clr')
#fig_shakirova_all_components = make_shakirova_component_figure('all')
#fig_shakirova_clr_components = make_shakirova_component_figure('clr')
#fig_shakirova_all_total_arctic = make_shakirova_total_figure('all', arctic=True)
#fig_shakirova_clr_total_arctic = make_shakirova_total_figure('clr', arctic=True)
#fig_shakirova_all_components_arctic = make_shakirova_component_figure('all', arctic=True)
#fig_shakirova_clr_components_arctic = make_shakirova_component_figure('clr', arctic=True)

In [ ]:
ablation_specs = [
    ("kernel", None, None, "kernel"),
    (
        "NN",
        Path("configs/model/fal/2011-2014_3,6,9,12_baseline.yaml"),
        None,
        "nn",
    ),
    (
        "NN+clear-sky",
        Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_clearsky.yaml"),
        Path("checkpoints/fal/2011-2014_3,6,9,12_baseline_clearsky/best_model.pt"),
        "nn",
    ),
    (
        "NN+clear-sky+Sob.",
        Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_clearsky.yaml"),
        Path("checkpoints/fal/2011-2014_3,6,9,12_sob_fal_clearsky/best_model.pt"),
        "nn",
    ),
]


def load_ablation_responses(label, config_path, checkpoint_path, source):
    if source == "kernel":
        return shakirova_responses

    ablation_config = load_config(config_path)
    ablation_vc = variable_config_from_omegaconf(ablation_config)
    checkpoint_path = checkpoint_path or Path(ablation_config.train.checkpoint_dir) / "best_model.pt"
    cache_path = shakirova_output_dir / f"ablation_responses_{checkpoint_path.parent.name}.nc"
    if cache_path.exists():
        return xr.load_dataset(cache_path)

    ablation_model, ablation_preprocessor, _ = load_model_and_preprocessor(
        ablation_config, checkpoint_path, downscaling=False
    )
    responses = compute_responses(
        shakirova_perturbed,
        shakirova_base,
        ablation_model,
        ablation_preprocessor,
        ablation_vc,
        shakirova_base_date,
        shakirova_perturbed_date,
        Path(ablation_config.dataset.era5.raw_path),
    )
    responses.to_netcdf(cache_path)
    return responses


def ablation_residuals(responses, source):
    all_terms = shakirova_terms(responses, "all")
    clr_terms = shakirova_terms(responses, "clr")
    if source == "kernel":
        return [all_terms["kernel"][-1], clr_terms["kernel"][-1], None, None]
    return [all_terms["nn"][-1], clr_terms["nn"][-1], all_terms["allcross"][-1], clr_terms["allcross"][-1]]

def coupling_slug(label):
    return "".join(ch if ch.isalnum() else "_" for ch in str(label).lower()).strip("_")

def load_coupling_model(config_path, checkpoint_path):
    coupling_config = load_config(config_path)
    coupling_vc = variable_config_from_omegaconf(coupling_config)
    checkpoint_path = checkpoint_path or Path(coupling_config.train.checkpoint_dir) / "best_model.pt"
    coupling_model, coupling_preprocessor, _ = load_model_and_preprocessor(
        coupling_config, checkpoint_path, downscaling=False
    )
    return coupling_model, coupling_preprocessor, coupling_vc

def albedo_cloud_coupling(label, config_path, checkpoint_path):
    cache_path = shakirova_output_dir / f"{coupling_slug(label)}_albedo_cloud_coupling.nc"
    if cache_path.exists():
        return xr.load_dataarray(cache_path)

    coupling_model, coupling_preprocessor, coupling_vc = load_coupling_model(config_path, checkpoint_path)
    cloud_vars = [var for var in coupling_vc.clear_sky_zero_vars if var in shakirova_base]
    states = xr.concat(
        [
            shakirova_base,
            shakirova_base.assign(fal=shakirova_perturbed.fal),
            shakirova_base.assign({var: shakirova_perturbed[var] for var in cloud_vars}),
            shakirova_base.assign(
                {"fal": shakirova_perturbed.fal, **{var: shakirova_perturbed[var] for var in cloud_vars}}
            ),
        ],
        dim="perturbation",
    )
    response = nn_response_for_states(
        states,
        ["perturbation", "latitude", "longitude"],
        coupling_model,
        coupling_preprocessor,
        coupling_vc,
    )
    coupling = response.sel(perturbation=3) - response.sel(perturbation=1) - response.sel(perturbation=2) + response.sel(perturbation=0)
    coupling.to_netcdf(cache_path)
    return coupling

def nn_response_for_states(states, dim_names, coupling_model, coupling_preprocessor, coupling_vc):
    processed = coupling_preprocessor.transform(states)
    return nn_pred(processed, coupling_model, coupling_preprocessor, coupling_vc, dim_names) / SECONDS_PER_DAY


def ablation_albedo_cloud_fields(label, config_path, checkpoint_path):
    responses = load_ablation_responses(label, config_path, checkpoint_path, "nn")
    terms = shakirova_terms(responses, "all")
    allcross_residual = terms["allcross"][-1]
    sum_residual = terms["nn"][-1]
    residual_diff = sum_residual - allcross_residual
    coupling = albedo_cloud_coupling(label, config_path, checkpoint_path)
    return [sum_residual, allcross_residual, residual_diff, coupling]


def field_correlation(a, b, arctic=True):
    if arctic:
        a, b = arctic_field(a), arctic_field(b)
    a_values, b_values = xr.align(a, b, join="inner")
    a_values = a_values.to_numpy().ravel()
    b_values = b_values.to_numpy().ravel()
    valid = np.isfinite(a_values) & np.isfinite(b_values)
    return float(np.corrcoef(a_values[valid], b_values[valid])[0, 1])

def make_ablation_albedo_cloud_bottom_row_figure(arctic=True):
    label, config_path, checkpoint_path, _ = ablation_specs[-1]
    responses = load_ablation_responses(label, config_path, checkpoint_path, "nn")
    terms = shakirova_terms(responses, "all")
    target = terms["target"]
    a_field = target - terms["allcross"][0]
    b_field = target - terms["nn"][-2]
    c_field = terms["kernel"][-1]
    d_field = b_field - a_field
    e_field = albedo_cloud_coupling(label, config_path, checkpoint_path)
    f_field = e_field - d_field

    panels = [
        (0, 0, a_field, r"$\Delta R - \Delta R_{\text{allcross}}^{NN}$", "a", -8, 8, 1, None, None),
        (1, 0, b_field, r"$\Delta R - \Delta R_{a+c+q}^{NN}$", "b", -8, 8, 1, None, None),
        (2, 0, c_field, r"$\Delta R - \Delta R_{a+c+q}^{K}$", "c", -8, 8, 1, None, None),
        (0, 1, d_field, r"(b) - (a)", "d", -8, 8, 1, None, None),
        (1, 1, e_field, r"$\Delta R_{ac}^{NN}$", "e", -8, 8, 1, None, None),
        (2, 1, f_field, r"(e) - (d)", "f", -8, 8, 1, None, None),
        (0, 2, shakirova_anomaly.fal, r"$\Delta a$", "g", -0.5, 0.5, 0.05, "", ""),
        (1, 2, shakirova_anomaly_ecod, r"$\Delta c$", "h", -1, 1, 0.1, "", ""),
        (2, 2, shakirova_anomaly.tcwv, r"$\Delta q$", "i", -20, 20, 2, "kg m$^{-2}$", "kg m$^{-2}$"),
    ]

    fig, ax = plt.subplots(
        3,
        3,
        figsize=(9, 9.5),
        subplot_kw=dict(
            projection=(
                ccrs.NorthPolarStereo(central_longitude=0)
                if arctic
                else ccrs.PlateCarree(central_longitude=180)
            )
        ),
        layout="constrained",
    )
    fig.get_layout_engine().set(rect=[0.06, 0.10, 1, 0.94], h_pad=0.01, w_pad=0.01, hspace=0.01, wspace=0.01)
    response_im = None
    response_axes = []
    for row, col, field, title, letter, vmin, vmax, step, cbar_label, units in panels:
        im = plot_shakirova_field(
            ax[row][col],
            field,
            panel_title(letter, title),
            vmin=vmin,
            vmax=vmax,
            step=step,
            left_labels=(col == 0),
            bottom_labels=(row == 2),
            arctic=arctic,
        )
        if cbar_label is None:
            response_im = im
            response_axes.append(ax[row][col])
            annotate_response(ax[row][col], field, residual=False, arctic=arctic)
        else:
            annotate_response(ax[row][col], field, residual=False, units=units)
            fig.colorbar(im, ax=ax[row][col], orientation="vertical", label=cbar_label)

    add_bottom_colorbar(fig, response_im, response_axes, "W m$^{-2}$", height=0.015)
    fig.set_layout_engine("none")
    region = "_arctic" if arctic else ""
    fig.savefig(shakirova_output_dir / f"shakirova_ablation_albedo_cloud_residuals_bottom_row{region}.png", bbox_inches="tight")
    return fig


In [ ]:
fig_shakirova_ablation_albedo_cloud_residuals_arctic = make_ablation_albedo_cloud_bottom_row_figure(arctic=True)

In [ ]:
def make_ablation_albedo_stack_figure(arctic=True):
    label, config_path, checkpoint_path, _ = ablation_specs[-1]
    responses = load_ablation_responses(label, config_path, checkpoint_path, "nn")
    terms = shakirova_terms(responses, "all")
    fields = [
        (terms["target"], r"$\Delta R$"),
        (terms["nn"][0], r"$\Delta R_a^{NN}$"),
        (terms["kernel"][0], r"$\Delta R_a^K$"),
    ]
    fig, ax = plt.subplots(
        3,
        1,
        figsize=(3.8, 8.4),
        subplot_kw=dict(
            projection=(
                ccrs.NorthPolarStereo(central_longitude=0)
                if arctic
                else ccrs.PlateCarree(central_longitude=180)
            )
        ),
        layout="constrained",
    )
    fig.get_layout_engine().set(rect=[0.10, 0.10, 0.94, 0.94], h_pad=0.01, w_pad=0.01, hspace=0.01, wspace=0.01)
    im = None
    for row, (field, title) in enumerate(fields):
        im = plot_shakirova_field(
            ax[row],
            field,
            panel_title("abc"[row], title),
            vmin=-30,
            vmax=30,
            step=5,
            left_labels=True,
            bottom_labels=(row == len(fields) - 1),
            arctic=arctic,
        )
        annotate_response(ax[row], field, residual=False, arctic=arctic)
    add_bottom_colorbar(fig, im, list(ax), "W m$^{-2}$", height=0.015)
    fig.set_layout_engine("none")
    region = "_arctic" if arctic else ""
    fig.savefig(shakirova_output_dir / f"shakirova_ablation_albedo_stack{region}.png", bbox_inches="tight")
    return fig


In [ ]:
make_ablation_albedo_stack_figure()

In [ ]:
def make_albedo_sensitivity_difference_figure(base_date=shakirova_base_date, perturbed_date=shakirova_perturbed_date, arctic=True):
    label, config_path, checkpoint_path, _ = ablation_specs[-1]
    cache_path = shakirova_output_dir / f"{coupling_slug(label)}_albedo_sensitivity_differences_{perturbed_date}_minus_{base_date}.nc"

    coupling_config = load_config(config_path)
    years = sorted({pd.Timestamp(base_date).year, pd.Timestamp(perturbed_date).year})
    raw_paths = generate_paths_yearly(coupling_config.dataset.era5.raw_path, years, make_era5_filename)
    with xr.open_mfdataset(raw_paths, combine="nested", concat_dim="date") as raw_for_dates:
        ds_for_dates = raw_for_dates.sel(date=[base_date, perturbed_date]).interp(**shakirova_grid).load()
    base = select_date_state(ds_for_dates, base_date)
    perturbed = select_date_state(ds_for_dates, perturbed_date)
    anomaly = perturbed - base

    coupling_model, coupling_preprocessor, coupling_vc = load_coupling_model(config_path, checkpoint_path)
    cloud_vars = [var for var in coupling_vc.clear_sky_zero_vars if var in base]
    delta_ecod = (coupling_preprocessor.transform(perturbed) - coupling_preprocessor.transform(base)).ecod.compute()

    if cache_path.exists():
        ds = xr.load_dataset(cache_path)
    else:
        def albedo_sensitivity(state):
            kernel, lon, lat = compute_nn_kernel_autograd(
                state.expand_dims(date=[base_date]),
                coupling_preprocessor,
                coupling_model,
                coupling_vc,
                var="fal",
            )
            return xr.DataArray(kernel, coords={"latitude": lat, "longitude": lon}, dims=("latitude", "longitude"))

        base_kernel = albedo_sensitivity(base)
        states = {
            "a": base.assign(fal=perturbed.fal),
            "c": base.assign({var: perturbed[var] for var in cloud_vars}),
            "q": base.assign(tcwv=perturbed.tcwv),
            "all": perturbed,
        }
        ds = xr.Dataset({name: albedo_sensitivity(state) - base_kernel for name, state in states.items()})
        ds.to_netcdf(cache_path)

    def true_albedo_kernel(date):
        kernels = xr.open_dataset(Path("data/fal/kernels") / make_kernel_filename(pd.Timestamp(date).year, "fal"))
        try:
            return (kernels.sel(date=date).fal.squeeze("date", drop=True).sel(all_clr="all") * kernel_delta("fal")).load()
        finally:
            kernels.close()

    true_delta_k = true_albedo_kernel(perturbed_date) - true_albedo_kernel(base_date)
    panels = [
        (0, 0, anomaly.fal, r"$\Delta a$", "a", -0.5, 0.5, 0.05, ""),
        (1, 0, delta_ecod, r"$\Delta c$", "b", -1, 1, 0.1, ""),
        (2, 0, anomaly.tcwv, r"$\Delta q$", "c", -8, 8, 2, "kg m$^{-2}$"),
        (0, 1, ds["a"], r"$\Delta K_a^{NN}$", "d", -0.6, 0.6, 0.1, None),
        (1, 1, ds["c"], r"$\Delta K_c^{NN}$", "e", -0.6, 0.6, 0.1, None),
        (2, 1, ds["q"], r"$\Delta K_q^{NN}$", "f", -0.6, 0.6, 0.1, None),
        (0, 2, true_delta_k, r"$\Delta K^{RRTM}$", "h", -0.6, 0.6, 0.1, None),
        (1, 2, ds["all"], r"$\Delta K^{NN}$", "g", -0.6, 0.6, 0.1, None),
        (2, 2, true_delta_k - ds["all"], r"$\Delta K^{RRTM} - \Delta K^{NN}$", "i", -0.6, 0.6, 0.1, None),
    ]

    fig, ax = plt.subplots(
        3,
        3,
        figsize=(11.69, 15),
        subplot_kw=dict(
            projection=(
                ccrs.NorthPolarStereo(central_longitude=0)
                if arctic
                else ccrs.PlateCarree(central_longitude=180)
            )
        ),
        layout="constrained",
    )
    fig.get_layout_engine().set(rect=[0.06, 0.08, 1, 0.94], h_pad=0.01, w_pad=0.01, hspace=0.01, wspace=0.01)

    kernel_im = None
    kernel_axes = []
    for row, col, field, title, letter, vmin, vmax, step, cbar_label in panels:
        im = plot_shakirova_field(
            ax[row][col],
            field,
            panel_title(letter, title),
            vmin=vmin,
            vmax=vmax,
            step=step,
            left_labels=(col == 0),
            bottom_labels=(row == 2),
            arctic=arctic,
            extend=None if col == 0 else 'both'
        )
        if cbar_label is None:
            kernel_im = im
            kernel_axes.append(ax[row][col])
            annotate_response(ax[row][col], field, residual=False, arctic=arctic, units="W m$^{-2}$ %$^{-1}$")
        else:
            fig.colorbar(im, ax=ax[row][col], orientation="horizontal", label=cbar_label)

    add_bottom_colorbar(fig, kernel_im, kernel_axes, "W m$^{-2}$ %$^{-1}$", height=0.02)
    fig.set_layout_engine("none")
    region = "_arctic" if arctic else ""
    fig.savefig(shakirova_output_dir / f"shakirova_albedo_sensitivity_differences_{perturbed_date}_minus_{base_date}{region}.png", bbox_inches="tight")
    return fig

def make_albedo_kernel_base_perturbed_figure(base_date=shakirova_base_date, perturbed_date=shakirova_perturbed_date, arctic=True):
    label, config_path, checkpoint_path, _ = ablation_specs[-1]
    cache_path = shakirova_output_dir / f"{coupling_slug(label)}_albedo_kernels_{base_date}_{perturbed_date}.nc"

    def true_albedo_kernel(date):
        kernels = xr.open_dataset(Path("data/fal/kernels") / make_kernel_filename(pd.Timestamp(date).year, "fal"))
        try:
            return (kernels.sel(date=date).fal.squeeze("date", drop=True).sel(all_clr="all") * kernel_delta("fal")).load()
        finally:
            kernels.close()

    if cache_path.exists():
        ds = xr.load_dataset(cache_path)
    else:
        coupling_config = load_config(config_path)
        years = sorted({pd.Timestamp(base_date).year, pd.Timestamp(perturbed_date).year})
        raw_paths = generate_paths_yearly(coupling_config.dataset.era5.raw_path, years, make_era5_filename)
        with xr.open_mfdataset(raw_paths, combine="nested", concat_dim="date") as raw_for_dates:
            ds_for_dates = raw_for_dates.sel(date=[base_date, perturbed_date]).interp(**shakirova_grid).load()
        coupling_model, coupling_preprocessor, coupling_vc = load_coupling_model(config_path, checkpoint_path)

        def nn_albedo_kernel(date):
            state = select_date_state(ds_for_dates, date)
            kernel, lon, lat = compute_nn_kernel_autograd(
                state.expand_dims(date=[date]),
                coupling_preprocessor,
                coupling_model,
                coupling_vc,
                var="fal",
            )
            return xr.DataArray(kernel, coords={"latitude": lat, "longitude": lon}, dims=("latitude", "longitude"))

        ds = xr.Dataset({
            "rrtm_base": true_albedo_kernel(base_date),
            "rrtm_perturbed": true_albedo_kernel(perturbed_date),
            "nn_base": nn_albedo_kernel(base_date),
            "nn_perturbed": nn_albedo_kernel(perturbed_date),
        })
        ds.to_netcdf(cache_path)

    kernel_panels = [
        (0, 0, ds["rrtm_base"], r"$K^{RRTM}$", "a"),
        (0, 1, ds["nn_base"], r"$K^{NN}$", "b"),
        (1, 0, ds["rrtm_perturbed"], r"$K'^{RRTM}$", "d"),
        (1, 1, ds["nn_perturbed"], r"$K'^{NN}$", "e"),
    ]
    residual_panels = [
        (0, 2, ds["rrtm_base"] - ds["nn_base"], r"$K^{RRTM} - K^{NN}$", "c"),
        (1, 2, ds["rrtm_perturbed"] - ds["nn_perturbed"], r"$K'^{RRTM} - K'^{NN}$", "f"),
    ]
    fig, ax = plt.subplots(
        2,
        3,
        figsize=(12, 9),
        subplot_kw=dict(
            projection=(
                ccrs.NorthPolarStereo(central_longitude=0)
                if arctic
                else ccrs.PlateCarree(central_longitude=180)
            )
        ),
        layout="constrained",
    )
    fig.get_layout_engine().set(rect=[0.06, 0.14, 1, 0.94], h_pad=0.01, w_pad=0.01, hspace=0.01, wspace=0.01)
    kernel_im = residual_im = None
    for row, col, field, title, letter in kernel_panels:
        kernel_im = plot_shakirova_field(
            ax[row][col],
            field,
            panel_title(letter, title),
            vmin=-2.6,
            vmax=0,
            step=0.2,
            left_labels=(col == 0),
            bottom_labels=(row == 1),
            arctic=arctic,
            cmap='Blues_r',
            extend='both'
        )
        annotate_response(ax[row][col], field, residual=False, arctic=arctic, units="W m$^{-2}$ %$^{-1}$")
    for row, col, field, title, letter in residual_panels:
        residual_im = plot_shakirova_field(
            ax[row][col],
            field,
            panel_title(letter, title),
            vmin=-0.4,
            vmax=0.4,
            step=0.1,
            left_labels=False,
            bottom_labels=(row == 1),
            arctic=arctic,
            extend='both'
        )
        annotate_response(ax[row][col], field, residual=False, arctic=arctic, units="W m$^{-2}$ %$^{-1}$")

    add_bottom_colorbar(fig, kernel_im, [ax[0][0], ax[0][1], ax[1][0], ax[1][1]], "W m$^{-2}$ %$^{-1}$", gap=0.01)
    add_bottom_colorbar(fig, residual_im, [ax[0][2], ax[1][2]], "W m$^{-2}$ %$^{-1}$", gap=0.01)
    fig.set_layout_engine("none")
    region = "_arctic" if arctic else ""
    fig.savefig(shakirova_output_dir / f"shakirova_albedo_kernels_{perturbed_date}_and_{base_date}{region}.png", bbox_inches="tight")
    return fig


In [ ]:
make_albedo_kernel_base_perturbed_figure(base_date='2012-08', perturbed_date='2015-08', arctic=True)

In [ ]:
make_albedo_sensitivity_difference_figure(base_date='2012-08', perturbed_date='2015-08', arctic=True)

In [ ]:
import pandas as pd

MAX_VALIDATION_POINTS = 4  # number of (date, sky) validation snapshots to keep per model/test

table_output_dir = shakirova_output_dir / "ablation_tables"
table_output_dir.mkdir(exist_ok=True, parents=True)

validation_output_dir = table_output_dir / "validation_points"
validation_output_dir.mkdir(exist_ok=True, parents=True)

neurips_qt_path = Path("data/era5/era5_plev_qt_monthly_downscaled_neurips_1990_2020.nc")

tsr_table_path = table_output_dir / "tsr_test_1990_2020.csv"
kernel_table_path = table_output_dir / "kernel_date_test_2015_heldout_months.csv"
kernel_fd_table_path = table_output_dir / "kernel_finite_difference_date_test_2015_heldout_months.csv"
closure_table_path = table_output_dir / "closure_1990_2020.csv"
feedback_table_path = table_output_dir / "feedback_quantification_two_year.csv"

# (display label, config path, checkpoint path or None -> use config's checkpoint_dir/best_model.pt)
table_model_specs = [
    #("NN", Path("configs/model/fal/2011-2014_3,6,9,12_baseline.yaml"), None),
    #("NN + clear-sky", Path("configs/model/fal/2011-2014_3,6,9,12_baseline_clearsky.yaml"), None),
    ("NN + clear-sky + Sob.", Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_4_clearsky.yaml"), None),
]

# Models for which we look for multi-seed checkpoints and report mean +/- std across seeds.
seed_aggregated_models = ["NN", "NN + clear-sky", "NN + clear-sky + Sob."]

def seed_checkpoint_paths(config_path, checkpoint_path):
    config_for_path = load_config(config_path)
    base_dir = checkpoint_path.parent if checkpoint_path is not None else Path(config_for_path.train.checkpoint_dir)
    return sorted(base_dir.parent.glob(f"{base_dir.name}_seed_*/best_model.pt"))

def expanded_table_model_specs():
    for label, config_path, checkpoint_path in table_model_specs:
        seeds = seed_checkpoint_paths(config_path, checkpoint_path) if label in seed_aggregated_models else []
        if len(seeds) > 1:
            for i, seed_path in enumerate(seeds, start=1):
                yield f"{label} [seed {i}]", config_path, seed_path
        else:
            yield label, config_path, checkpoint_path
        #yield label, config_path, checkpoint_path # simple for now


def seed_rows_for(table, label):
    """Boolean mask selecting rows of `table` belonging to seed variants of `label`."""
    index = table.index.get_level_values(0) if isinstance(table.index, pd.MultiIndex) else table.index
    return index.astype(str).str.startswith(f"{label} [seed ")


def format_mean_std(mean, std):
    return mean.combine(std, lambda m, s: f"{m:.3f} \u00b1 {s:.3f}")


def aggregate_seed_rows(table):
    #print(table)
    """Collapse per-seed rows into a single mean +/- std row per seed-aggregated model."""
    multi = isinstance(table.index, pd.MultiIndex)
    rows, seen_mask = [], pd.Series(False, index=table.index)

    for label in seed_aggregated_models:
        mask = seed_rows_for(table, label)
        seen_mask |= mask
        if not mask.any():
            continue
        seed_table = table.loc[mask].astype(float)

        if not multi:
            rows.append(pd.DataFrame([format_mean_std(seed_table.mean(), seed_table.std(ddof=1))], index=[label]))
            continue

        for residual in seed_table.index.get_level_values(1).unique():
            residual_table = seed_table.xs(residual, level=1)
            row = format_mean_std(residual_table.mean(), residual_table.std(ddof=1))
            row.name = (label, residual)
            rows.append(row)

    if not rows:
        return table

    keep = table.loc[~seen_mask.to_numpy()]
    if not multi:
        return pd.concat([*rows, keep])

    aggregated = pd.DataFrame(rows)
    aggregated.index = pd.MultiIndex.from_tuples(aggregated.index, names=table.index.names)
    return pd.concat([aggregated, keep])


def seed_aggregation_available():
    """Whether any seed-aggregated model currently has >1 discovered seed checkpoint."""
    return any(
        len(seed_checkpoint_paths(config_path, checkpoint_path)) > 1
        for label, config_path, checkpoint_path in table_model_specs
        if label in seed_aggregated_models
    )


def table_has_seed_aggregation(table):
    """Whether a cached table already contains mean +/- std formatted seed rows."""
    values = table.astype(str).to_numpy().ravel()
    return any("\u00b1" in value for value in values)


def safe_name(value):
    """Filesystem-safe version of a label, for use in output filenames."""
    return "".join(ch if ch.isalnum() else "_" for ch in str(value)).strip("_")

def load_table_model(config_path, checkpoint_path):
    table_config = load_config(config_path)
    table_vc = variable_config_from_omegaconf(table_config)
    checkpoint_path = checkpoint_path or Path(table_config.train.checkpoint_dir) / "best_model.pt"
    table_model, table_preprocessor, _ = load_model_and_preprocessor(
        table_config, checkpoint_path, downscaling=False
    )
    return table_config, table_vc, table_model, table_preprocessor


def model_rows(row_fn, *args, **kwargs):
    """Load each model spec and collect the rows `row_fn` produces for it."""
    rows = []
    for label, config_path, checkpoint_path in expanded_table_model_specs():
        table_config, table_vc, table_model, table_preprocessor = load_table_model(config_path, checkpoint_path)
        rows.extend(row_fn(label, table_config, table_vc, table_model, table_preprocessor, *args, **kwargs))
    return rows


# --------------------------------------------------------------------------- #
# Validation-point snapshots (for spot-checking predictions vs truth)
# --------------------------------------------------------------------------- #

def validation_point_dataset(test, model, date, sky, prediction, truth):
    prediction = prediction.reset_coords(drop=True).transpose("latitude", "longitude").astype("float32")
    truth = truth.reset_coords(drop=True).transpose("latitude", "longitude").astype("float32")
    return xr.Dataset(
        {
            "truth": truth.expand_dims(record=1),
            "prediction": prediction.expand_dims(record=1),
            "residual": (prediction - truth).astype("float32").expand_dims(record=1),
        },
        coords={
            "test": ("record", [test]),
            "model": ("record", [model]),
            "date": ("record", [pd.Timestamp(date)]),
            "sky": ("record", [sky]),
        },
    )


def write_validation_points(path, datasets):
    if not datasets:
        return
    ds = xr.concat(datasets, dim="record").assign_coords(record=np.arange(len(datasets)))
    ds.to_netcdf(path)


# --------------------------------------------------------------------------- #
# Metric bookkeeping (mean bias, RMSE, across-model summary tables)
# --------------------------------------------------------------------------- #

def row_metric(diff, arctic=False):
    #print(diff)
    
    field = arctic_field(diff) if arctic else diff
    mean = float(global_mean(field).compute())
    mean_sq = float(global_mean(field**2).compute())
    return mean, mean_sq


def add_metric(rows, model, sky, diff, arctic=False):
    mean, mean_sq = row_metric(diff, arctic=arctic)
    rows.append({"Model": model, "sky": sky, "mean": mean, "mean_sq": mean_sq})


def summarize_metric_rows(rows, include_std=True):
    rows_df = pd.DataFrame(rows)
    table_rows = []
    for model, model_df in rows_df.groupby("Model", sort=False):
        row = {"Model": model}
        for sky in ["all", "clear"]:
            sky_df = model_df[model_df.sky == sky]
            row[f"{sky} MBE"] = float(sky_df["mean"].mean())
            row[f"{sky} RMSE"] = float(np.sqrt(sky_df["mean_sq"].mean()))
            if include_std:
                row[f"{sky} STD"] = float(sky_df["mean"].std(ddof=0))
        table_rows.append(row)
    return pd.DataFrame(table_rows).set_index("Model")


def add_closure_metric(rows, model, residual, sky, region, diff):
    mean, mean_sq = row_metric(diff, arctic=(region == "arctic"))
    rows.append({"Model": model, "Residual": residual, "sky": sky, "region": region, "mean": mean, "mean_sq": mean_sq})


def summarize_closure_metric_rows(rows):
    rows_df = pd.DataFrame(rows)
    table_rows = []
    for (model, residual), group_df in rows_df.groupby(["Model", "Residual"], sort=False):
        row = {"Model": model, "Residual": residual}
        for region in ["global", "arctic"]:
            for sky in ["all", "clear"]:
                subset = group_df[(group_df.region == region) & (group_df.sky == sky)]
                row[f"{region} {sky} MBE"] = float(subset["mean"].mean())
                row[f"{region} {sky} RMSE"] = float(np.sqrt(subset["mean_sq"].mean()))
        table_rows.append(row)
    return pd.DataFrame(table_rows).set_index(["Model", "Residual"])


# --------------------------------------------------------------------------- #
# TSR test: NN TSR predictions vs ERA5 TSR, 1990-2020, all-sky & clear-sky
# --------------------------------------------------------------------------- #

def tsr_metric_rows(label, table_config, table_vc, table_model, table_preprocessor):
    rows, validation_points = [], []
    for year in range(1990, 2021):
        raw_ds = xr.open_dataset(Path(table_config.dataset.era5.raw_path) / make_era5_filename(year))
        for date in raw_ds.date.values:
            raw_date = raw_ds.sel(date=date)

            all_pred = nn_pred(
                table_preprocessor.transform(raw_date),
                table_model, table_preprocessor, table_vc,
                ["latitude", "longitude"],
            ) / SECONDS_PER_DAY
            clear_pred = nn_pred(
                table_preprocessor.transform(table_vc.clear_sky_input(raw_date)),
                table_model, table_preprocessor, table_vc,
                ["latitude", "longitude"],
                clear=True,
            ) / SECONDS_PER_DAY

            all_truth = raw_date.tsr / SECONDS_PER_DAY
            clear_truth = raw_date.tsrc / SECONDS_PER_DAY

            add_metric(rows, label, "all", all_pred - all_truth)
            add_metric(rows, label, "clear", clear_pred - clear_truth)

            if len(validation_points) < MAX_VALIDATION_POINTS:
                validation_points.extend([
                    validation_point_dataset("tsr", label, date, "all", all_pred, all_truth),
                    validation_point_dataset("tsr", label, date, "clear", clear_pred, clear_truth),
                ])
        raw_ds.close()

    write_validation_points(validation_output_dir / f"tsr_{safe_name(label)}.nc", validation_points)
    return rows


def save_table(table, path):
    table.to_csv(path)
    return table


def compute_tsr_table():
    table = aggregate_seed_rows(summarize_metric_rows(model_rows(tsr_metric_rows), include_std=True))
    return save_table(table, tsr_table_path)


# --------------------------------------------------------------------------- #
# Kernel date test: 2015 held-out months, NN albedo kernel vs FAL kernel
# (autograd-derived and, optionally, finite-difference-derived)
# --------------------------------------------------------------------------- #

def kernel_metric_rows(label, table_config, table_vc, table_model, table_preprocessor, finite_difference=False):
    rows, validation_points = [], []
    held_out_months = [month for month in range(1, 13) if month not in (3, 6, 9, 12)]
    dates = [f"2015-{month:02d}" for month in held_out_months]

    raw_ds = xr.open_dataset(Path(table_config.dataset.era5.raw_path) / make_era5_filename(2015))
    kernels = xr.open_dataset(Path(table_config.dataset.kernels.raw_path) / make_kernel_filename(2015, "fal"))

    for date in dates:
        raw_date = (
            raw_ds.sel(date=[date] if finite_difference else date)
            .interp(latitude=kernels.latitude, longitude=kernels.longitude)
        )
        true_kernel = kernels.sel(date=date).fal.squeeze("date", drop=True) * kernel_delta("fal")

        if finite_difference:
            pred_all, lon, lat = compute_nn_kernel(
                raw_date, table_preprocessor, table_model, table_vc, perturbation_var="fal",
            )
            pred_clear, _, _ = compute_nn_kernel(
                raw_date, table_preprocessor, table_model, table_vc, clear=True, perturbation_var="fal",
            )
        else:
            pred_all, lon, lat = compute_nn_kernel_autograd(
                raw_date, table_preprocessor, table_model, table_vc, var="fal", clear=False,
            )
            pred_clear, _, _ = compute_nn_kernel_autograd(
                raw_date, table_preprocessor, table_model, table_vc, var="fal", clear=True,
            )

        all_pred = xr.DataArray(pred_all, coords={"latitude": lat, "longitude": lon}, dims=("latitude", "longitude"))
        clear_pred = xr.DataArray(pred_clear, coords={"latitude": lat, "longitude": lon}, dims=("latitude", "longitude"))
        all_truth = true_kernel.sel(all_clr="all")
        clear_truth = true_kernel.sel(all_clr="clr")

        add_metric(rows, label, "all", all_pred - all_truth)
        add_metric(rows, label, "clear", clear_pred - clear_truth)

        if len(validation_points) < MAX_VALIDATION_POINTS:
            test_name = "kernel_finite_difference" if finite_difference else "kernel"
            validation_points.extend([
                validation_point_dataset(test_name, label, date, "all", all_pred, all_truth),
                validation_point_dataset(test_name, label, date, "clear", clear_pred, clear_truth),
            ])

    raw_ds.close()
    kernels.close()

    suffix = "kernel_finite_difference" if finite_difference else "kernel"
    write_validation_points(validation_output_dir / f"{suffix}_{safe_name(label)}.nc", validation_points)
    return rows


def compute_kernel_table():
    table = aggregate_seed_rows(summarize_metric_rows(model_rows(kernel_metric_rows), include_std=True))
    return save_table(table, kernel_table_path)


def compute_kernel_fd_table():
    rows = model_rows(kernel_metric_rows, finite_difference=True)
    table = aggregate_seed_rows(summarize_metric_rows(rows, include_std=True))
    return save_table(table, kernel_fd_table_path)

# --------------------------------------------------------------------------- #
# Feedback quantification test: shakirova_base_date -> shakirova_perturbed_date response residuals
# --------------------------------------------------------------------------- #

def feedback_metric_rows(label, config_path, checkpoint_path):
    rows = []
    responses = load_ablation_responses(label, config_path, checkpoint_path, "nn")
    for sky, response_sky in [("all", "all"), ("clear", "clr")]:
        terms = shakirova_terms(responses, response_sky)
        for residual, diff in [("allcross", terms["allcross"][-1]), ("nn sum", terms["nn"][-1])]:
            for region in ["global", "arctic"]:
                add_closure_metric(rows, label, residual, sky, region, diff)
    return rows

"""
def load_ablation_responses(label, config_path, checkpoint_path, source):
    if source == "kernel":
        return shakirova_responses

    ablation_config = load_config(config_path)
    ablation_vc = variable_config_from_omegaconf(ablation_config)
    checkpoint_path = checkpoint_path or Path(ablation_config.train.checkpoint_dir) / "best_model.pt"
    cache_path = shakirova_output_dir / f"ablation_responses_{checkpoint_path.parent.name}.nc"
    if cache_path.exists():
        return xr.load_dataset(cache_path)

    ablation_model, ablation_preprocessor, _ = load_model_and_preprocessor(
        ablation_config, checkpoint_path, downscaling=False
    )
    responses = compute_responses(
        shakirova_perturbed,
        shakirova_base,
        ablation_model,
        ablation_preprocessor,
        ablation_vc,
        shakirova_base_date,
        shakirova_perturbed_date,
        Path(ablation_config.dataset.era5.raw_path),
    )
    responses.to_netcdf(cache_path)
    return responses



def shakirova_terms(responses, sky):
    suffix = "all" if sky == "all" else "clr"
    target = responses[f"dR_era5_{suffix}"]
    if sky == "all":
        kernel_parts = [
            responses["dR_a_k_all"],
            responses["dR_q_k_all"],
            responses["dR_c_k_all"],
        ]
        nn_parts = [
            responses["dR_a_nn_all"],
            responses["dR_q_nn_all"],
            responses["dR_c_nn_all"],
        ]
        nn_allcross = responses["dR_nn_all"]
    else:
        kernel_parts = [responses["dR_a_k_clr"], responses["dR_q_k_clr"]]
        nn_parts = [responses["dR_a_nn_clr"], responses["dR_q_nn_clr"]]
        nn_allcross = responses["dR_nn_clr"]

    kernel_sum = add_fields(kernel_parts)
    nn_sum = add_fields(nn_parts)
    return {
        "target": target,
        "kernel": [*kernel_parts, kernel_sum, target - kernel_sum],
        "nn": [*nn_parts, nn_sum, target - nn_sum],
        "allcross": [nn_allcross, target - nn_allcross],
    }
"""

def feedback_kernel_metric_rows():
    rows = []
    responses = load_ablation_responses(None, None, None, "kernel")
    for sky, response_sky in [("all", "all"), ("clear", "clr")]:
        terms = shakirova_terms(responses, response_sky)
        diff = terms["kernel"][-1]
        for region in ["global", "arctic"]:
            add_closure_metric(rows, "Kernel", "kernel sum", sky, region, diff)
    return rows


def compute_feedback_table():
    rows = []
    for label, config_path, checkpoint_path in expanded_table_model_specs():
        rows.extend(feedback_metric_rows(label, config_path, checkpoint_path))
    rows.extend(feedback_kernel_metric_rows())

    table = aggregate_seed_rows(summarize_closure_metric_rows(rows))
    return save_table(table, feedback_table_path)


def ensure_feedback_kernel_method(table):
    """Backfill the kernel-method row into a cached feedback table that predates it."""
    if isinstance(table.index, pd.MultiIndex) and ("Kernel", "kernel sum") in table.index:
        return table
    print("Feedback quantification test: recomputing for method-level mean/std rows")
    return compute_feedback_table()


# --------------------------------------------------------------------------- #
# Cache loading and top-level table computation
# --------------------------------------------------------------------------- #

def load_or_compute_table(path, description, compute_table, index_col=0):
    """Load a cached CSV table if present and up to date, else compute and cache it."""
    if path.exists():
        table = pd.read_csv(path, index_col=index_col)
        if seed_aggregation_available() and not table_has_seed_aggregation(table):
            print(f"{description}: recomputing for seed mean/std")
            return compute_table()
        print(f"{description}: loaded {path}")
        return table
    print(f"{description}: computing")
    return compute_table()

"
tsr_ablation_table = load_or_compute_table(
    tsr_table_path,
    "TSR test: monthly 1990-2020 NN TSR predictions vs ERA5 TSR, all-sky and clear-sky",
    compute_tsr_table,
)
kernel_ablation_table = load_or_compute_table(
    kernel_table_path,
    "Kernel date test: 2015 held-out months except March, June, September, December; "
    "NN autograd albedo kernel vs FAL kernel",
    compute_kernel_table,
)
kernel_fd_ablation_table = load_or_compute_table(
    kernel_fd_table_path,
    "Kernel finite-difference date test: same 2015 held-out months; "
    "NN finite-difference albedo kernel vs FAL kernel",
    compute_kernel_fd_table,
)

feedback_ablation_table = ensure_feedback_kernel_method(load_or_compute_table(
    feedback_table_path,
    "Feedback quantification test: shakirova_base_date to shakirova_perturbed_date response residuals, global and Arctic",
    compute_feedback_table,
    index_col=[0, 1],
))

"""

In [ ]:
display(tsr_ablation_table.round(3))
display(kernel_ablation_table.round(3))
display(kernel_fd_ablation_table.round(3))
display(feedback_ablation_table.round(3))

In [ ]:
#fig_shakirova_ablation_residuals_arctic = make_ablation_residual_figure(arctic=True)

In [ ]:
import re

sobolev_lambda_specs = [
    ("lambda=0.1", Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_0.1_clearsky.yaml")),
    ("lambda=0.25", Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_0.25_clearsky.yaml")),
    ("lambda=0.5", Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_0.5_clearsky.yaml")),
    ("lambda=1", Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_1_clearsky.yaml")),
    ("lambda=2", Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_2_clearsky.yaml")),
    ("lambda=4", Path("configs/model/fal/2011-2014_3,6,9,12_sob_fal_4_clearsky.yaml")),
]

sobolev_lambda_tsr_raw_path = table_output_dir / "sobolev_lambda_seed_metrics_tsr_test_1990_2020.csv"
sobolev_lambda_kernel_raw_path = table_output_dir / "sobolev_lambda_seed_metrics_kernel_date_test_2015_heldout_months.csv"
sobolev_lambda_feedback_raw_path = table_output_dir / "sobolev_lambda_seed_metrics_feedback_quantification_two_year.csv"


def sobolev_lambda_seed_paths(label, config_path):
    seeds = seed_checkpoint_paths(config_path, None)
    if len(seeds) != 5:
        raise FileNotFoundError(f"{label}: expected 5 seeded checkpoints, found {len(seeds)}")
    return seeds


def sobolev_seed_labels(label):
    return [f"{label} [seed {i}]" for i in range(1, 6)]


def strip_sobolev_seed_label(value):
    return re.sub(r" \[seed \d+\](?=$| )", "", str(value))


def sobolev_seed_mean_std_table(table):
    table = table.astype(float).copy()
    if isinstance(table.index, pd.MultiIndex):
        tuples = [tuple([strip_sobolev_seed_label(parts[0]), *parts[1:]]) for parts in table.index]
        table.index = pd.MultiIndex.from_tuples(tuples, names=table.index.names)
        grouped = table.groupby(level=list(range(table.index.nlevels)), sort=False)
    else:
        table.index = table.index.map(strip_sobolev_seed_label)
        grouped = table.groupby(level=0, sort=False)

    mean = grouped.mean()
    std = grouped.std(ddof=1)
    return mean.map(lambda x: f"{x:.3f}") + " ± " + std.map(lambda x: f"{x:.3f}")


def missing_sobolev_specs(table):
    existing = set((table.index.get_level_values(0) if isinstance(table.index, pd.MultiIndex) else table.index).astype(str))
    return [(label, config_path) for label, config_path in sobolev_lambda_specs if not set(sobolev_seed_labels(label)).issubset(existing)]


def sobolev_lambda_model_rows(row_fn, specs=None, *args, **kwargs):
    rows = []
    for label, config_path in (specs or sobolev_lambda_specs):
        for seed_label, checkpoint_path in zip(sobolev_seed_labels(label), sobolev_lambda_seed_paths(label, config_path)):
            table_config, table_vc, table_model, table_preprocessor = load_table_model(config_path, checkpoint_path)
            rows.extend(row_fn(seed_label, table_config, table_vc, table_model, table_preprocessor, *args, **kwargs))
    return rows


def load_or_update_raw_sobolev_table(path, description, compute_table, index_col=0):
    if not path.exists():
        print(f"{description}: computing")
        table = compute_table(sobolev_lambda_specs)
        table.to_csv(path)
        return table

    table = pd.read_csv(path, index_col=index_col)
    missing = missing_sobolev_specs(table)
    if not missing:
        print(f"{description}: loaded {path}")
        return table

    print(f"{description}: computing missing configs {[label for label, _ in missing]}")
    table = pd.concat([table, compute_table(missing)])
    table = table[~table.index.duplicated(keep="last")]
    table.to_csv(path)
    return table


def compute_sobolev_lambda_tsr_raw_table(specs):
    return summarize_metric_rows(sobolev_lambda_model_rows(tsr_metric_rows, specs), include_std=False)


def compute_sobolev_lambda_kernel_raw_table(specs):
    return summarize_metric_rows(sobolev_lambda_model_rows(kernel_metric_rows, specs), include_std=False)


def compute_sobolev_lambda_feedback_raw_table(specs):
    rows = []
    for label, config_path in specs:
        for seed_label, checkpoint_path in zip(sobolev_seed_labels(label), sobolev_lambda_seed_paths(label, config_path)):
            rows.extend(feedback_metric_rows(seed_label, config_path, checkpoint_path))
    return summarize_closure_metric_rows(rows)


sobolev_lambda_tsr_raw_table = load_or_update_raw_sobolev_table(
    sobolev_lambda_tsr_raw_path,
    "Sobolev lambda ablation TSR seed metrics",
    compute_sobolev_lambda_tsr_raw_table,
)
sobolev_lambda_kernel_raw_table = load_or_update_raw_sobolev_table(
    sobolev_lambda_kernel_raw_path,
    "Sobolev lambda ablation kernel sensitivity seed metrics",
    compute_sobolev_lambda_kernel_raw_table,
)
sobolev_lambda_feedback_raw_table = load_or_update_raw_sobolev_table(
    sobolev_lambda_feedback_raw_path,
    "Sobolev lambda ablation two-year feedback seed metrics",
    compute_sobolev_lambda_feedback_raw_table,
    index_col=[0, 1],
)

sobolev_lambda_tsr_table = sobolev_seed_mean_std_table(sobolev_lambda_tsr_raw_table)
sobolev_lambda_kernel_table = sobolev_seed_mean_std_table(sobolev_lambda_kernel_raw_table)
sobolev_lambda_feedback_table = sobolev_seed_mean_std_table(sobolev_lambda_feedback_raw_table)

display(sobolev_lambda_tsr_raw_table)
display(sobolev_lambda_kernel_raw_table)
display(sobolev_lambda_feedback_raw_table)
display(sobolev_lambda_tsr_table)
display(sobolev_lambda_kernel_table)
display(sobolev_lambda_feedback_table)

# %%


In [ ]:
def make_neurips_albedo_sensitivity_figure(date="2015-07", arctic=False):
    output_path = shakirova_output_dir / f"neurips_albedo_radiative_sensitivity_{date}.png"
    raw_ds = xr.open_dataset(Path("data/era5") / make_era5_filename(pd.Timestamp(date).year))
    kernels = xr.open_dataset(Path("data/fal/kernels") / make_kernel_filename(pd.Timestamp(date).year, "fal"))
    kernel_grid = {"latitude": kernels.latitude, "longitude": kernels.longitude}
    raw_fd = raw_ds.sel(date=[date]).interp(**kernel_grid)
    raw_autograd = raw_ds.sel(date=[date]).interp(**kernel_grid)
    true_kernel = kernels.sel(date=date).fal.squeeze("date", drop=True) * kernel_delta("fal")

    def as_field(values, lon, lat):
        return xr.DataArray(values, coords={"longitude": lon, "latitude": lat}, dims=("latitude", "longitude"))

    def get_prediction(config, method="fd", clear=False):
        c, vc, m, pp = load_table_model(config, None)
        if method == "fd":
            pred, lon, lat = compute_nn_kernel(raw_fd, pp, m, vc, clear=clear, perturbation_var="fal")
        else:
            pred, lon, lat = compute_nn_kernel_autograd(raw_autograd, pp, m, vc, var="fal", clear=clear)
        return as_field(pred, lon, lat)

    rows = [
        ("RRTM", true_kernel.sel(all_clr="all"), true_kernel.sel(all_clr="clr")),
        (
            "NN",
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_baseline.yaml", method="fd", clear=False),
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_baseline.yaml", method="fd", clear=True),
        ),
        (
            "NN+clear-sky",
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_baseline_clearsky.yaml", method="fd", clear=False),
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_baseline_clearsky.yaml", method="fd", clear=True),
        ),
        (
            "NN+clear-sky+Sob.",
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_sob_fal_clearsky.yaml", method="autograd", clear=False),
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_sob_fal_clearsky.yaml", method="autograd", clear=True),
        ),
    ]
    fig, ax = plt.subplots(
        4,
        2,
        figsize=(11.69, 13.5),
        subplot_kw=dict(
            projection=(
                ccrs.NorthPolarStereo(central_longitude=0)
                if arctic
                else ccrs.PlateCarree(central_longitude=180)
            )
        ),
        layout="constrained",
    )
    fig.get_layout_engine().set(rect=[0.06, 0.10, 0.96, 0.94], h_pad=0.01, w_pad=0.01, hspace=0.01, wspace=0.01)
    im = None
    letters = iter("aebfcgdh")
    for row, (label, all_field, clear_field) in enumerate(rows):
        for col, (field, sky) in enumerate([(all_field, "all-sky"), (clear_field, "clear-sky")]):
            im = plot_shakirova_field(
                ax[row][col],
                field,
                panel_title(next(letters), f"{label}" + ((" " + sky) if row == 0 else "")),
                vmin=-3,
                vmax=3,
                step=0.2,
                left_labels=(col == 0),
                bottom_labels=(row == len(rows) - 1),
                arctic=arctic,
            )
            annotate_response(ax[row][col], field, arctic=arctic, units = "W m$^{-2}$ %$^{-1}$")

    add_right_colorbar(fig, im, ax.ravel().tolist(), "W m$^{-2}$ %$^{-1}$", width=0.03)
    fig.set_layout_engine("none")
    fig.savefig(output_path, bbox_inches="tight")
    raw_ds.close()
    kernels.close()
    return fig


fig_neurips_albedo_sensitivity = make_neurips_albedo_sensitivity_figure("2015-07", arctic=False)


In [ ]:
def make_tsi_albedo_sensitivity_figure(date="2015-07", arctic=False):
    Path('FiguresTSI').mkdir(exist_ok=True)
    output_path = Path('FiguresTSI') / f"albedo_radiative_sensitivity_{date}.png"
    plt.style.use('dark_background')
    raw_ds = xr.open_dataset(Path("data/era5") / make_era5_filename(pd.Timestamp(date).year))
    kernels = xr.open_dataset(Path("data/fal/kernels") / make_kernel_filename(pd.Timestamp(date).year, "fal"))
    kernel_grid = {"latitude": kernels.latitude, "longitude": kernels.longitude}
    raw_fd = raw_ds.sel(date=[date]).interp(**kernel_grid)
    raw_autograd = raw_ds.sel(date=[date]).interp(**kernel_grid)
    true_kernel = kernels.sel(date=date).fal.squeeze("date", drop=True) * kernel_delta("fal")

    def as_field(values, lon, lat):
        return xr.DataArray(values, coords={"longitude": lon, "latitude": lat}, dims=("latitude", "longitude"))

    def get_prediction(config, method="fd", clear=False):
        c, vc, m, pp = load_table_model(config, None)
        if method == "fd":
            pred, lon, lat = compute_nn_kernel(raw_fd, pp, m, vc, clear=clear, perturbation_var="fal")
        else:
            pred, lon, lat = compute_nn_kernel_autograd(raw_autograd, pp, m, vc, var="fal", clear=clear)
        return as_field(pred, lon, lat)

    rows = [
        ("RRTM", true_kernel.sel(all_clr="all"), true_kernel.sel(all_clr="clr")),
        (
            "NN",
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_baseline.yaml", method="fd", clear=False),
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_baseline.yaml", method="fd", clear=True),
        ),
        (
            "NN+clear-sky+Sob.",
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_sob_fal_clearsky.yaml", method="autograd", clear=False),
            get_prediction("configs/model/fal/2011-2014_3,6,9,12_sob_fal_clearsky.yaml", method="autograd", clear=True),
        ),
    ]
    fig, ax = plt.subplots(
        1,
        3,
        figsize=(11.69, 4),
        subplot_kw=dict(
            projection=(
                ccrs.NorthPolarStereo(central_longitude=0)
                if arctic
                else ccrs.PlateCarree(central_longitude=180)
            )
        ),
        layout="constrained",
        dpi=600
    )
    fig.get_layout_engine().set(rect=[0.06, 0.10, 0.96, 0.94], h_pad=0.01, w_pad=0.01, hspace=0.01, wspace=0.01)
    im = None
    letters = iter("abc")
    for col, (label, all_field, clear_field) in enumerate(rows):
            field = all_field.where(all_field <= 0, other=0)
            im = plot_shakirova_field(
                ax[col],
                field,
                panel_title(next(letters), f"{label}"),
                vmin=-3.6,
                vmax=0,
                step=0.2,
                left_labels=(col == 0),
                bottom_labels=True,
                arctic=arctic,
                cmap='Blues_r',
                extend=None
            )
            annotate_response(ax[col], field, arctic=arctic, units = "W m$^{-2}$ %$^{-1}$")

    add_bottom_colorbar(fig, im, ax.ravel().tolist(), "W m$^{-2}$ %$^{-1}$", height=0.05)
    fig.set_layout_engine("none")
    fig.savefig(output_path, bbox_inches="tight")
    raw_ds.close()
    kernels.close()
    return fig


make_tsi_albedo_sensitivity_figure("2015-07", arctic=False)
